# Gravitational Molecule v3: Single-Step Cloud Termination

This notebook extends the three-point non-Hermitian calculation completed in `Mathematica/v3` into one full iteration. Each computational stage has its own numbered code cell. Only one step from $R$ to $R-\Delta R$ is shown; no iteration loop is included. The total cloud rate is determined by $2\operatorname{Im}E$, and eigenvector weights distribute the signed mass exchange between the two black holes.

## Current Mathematica v3 coverage

The original v3 covers the Hamiltonian, growth widths, non-Hermitian eigensystem, biorthogonal cloud moments, GW frequency and power, cloud and system energies, and an experimental emission ratio. It does not yet include the orbital step duration, cloud decay, mass transfer back to both black holes, or the next state; the 15 steps below complete one iteration. Conservative orbital quantities explicitly use the real parts of biorthogonal results, while imaginary parts are used only for growth or decay.

In [27]:
# Step 1 — Imports and runtime environment
from pathlib import Path
import sys

import numpy as np

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (working_directory, *working_directory.parents)
    if (candidate / 'src' / 'gmlib.py').is_file()
)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from gmlib import (
    PLANCK_TIMES_PER_YEAR, IntegrationSettings, IterationState,
    apply_mass_exchange, build_nonhermitian_hamiltonians,
    compute_cloud_observables, compute_gr_power,
    compute_gw_frequency_squared, compute_iteration_tables,
    compute_mass_quadrupole, compute_signed_rates,
    compute_step_duration, compute_system_energy,
    diagonalize_and_track, make_stencil,
)

np.set_printoptions(precision=8, suppress=False)
PROJECT_ROOT

PosixPath('/global/u1/l/liuyh15/WorkSpace/GM')

In [28]:
# Step 2 — Current physical parameters (state 1 uses zero-based index 0 in Python)
m1_solar = 40.0
q = 0.99
alpha1 = 0.1
qc = 0.1
a1 = 1.0
a2 = 1.0
current_R = 38.0
selected_state = 0

R_start = 38.0
R_end = 10.0
number = 200  # Same as v2: number of grid points

{
    'M1 [solar]': m1_solar, 'q': q, 'alpha1': alpha1, 'qc': qc,
    'a1': a1, 'a2': a2, 'R': current_R, 'state': selected_state + 1,
}

{'M1 [solar]': 40.0,
 'q': 0.99,
 'alpha1': 0.1,
 'qc': 0.1,
 'a1': 1.0,
 'a2': 1.0,
 'R': 38.0,
 'state': 1}

In [29]:
# Step 3 — Match v2: define the step and three-point stencil directly from 200 grid points
R_values = np.linspace(R_end, R_start, number)
delta_R = R_values[1] - R_values[0]
state = IterationState.from_ratios(
    primary_mass_solar=m1_solar, mass_ratio=q, alpha_primary=alpha1,
    cloud_mass_fraction=qc, spin_primary=a1, spin_secondary=a2,
    separation=current_R, selected_state=selected_state,
)
stencil = make_stencil(state, delta_R)
{
    'number of grid points': R_values.size,
    'number of intervals': R_values.size - 1,
    'delta_R': delta_R, 'stencil': stencil, 'state': state.summary(),
}

{'number of grid points': 200,
 'number of intervals': 199,
 'delta_R': np.float64(0.14070351758793898),
 'stencil': array([37.85929648, 38.        , 38.14070352]),
 'state': {'M1_solar': 40.0,
  'M2_solar': 39.6,
  'Mc_solar': 4.0,
  'q': 0.99,
  'qc': 0.1,
  'alpha1': 0.09999999999999999,
  'alpha2': 0.099,
  'a1': 1.0,
  'a2': 1.0,
  'R': 38.0,
  'state': 1}}

In [30]:
# Step 4 — Compute H, dH/dOmega, omega Lz, and raw moment operators at three separations
integration = IntegrationSettings(workers=-1, verbose=True)
tables = compute_iteration_tables(state, delta_R, integration)
{
    'H': tables.hamiltonian_dimless.shape,
    'dH/dOmega': tables.dh_domega.shape,
    'omega Lz': tables.omega_lz_dimless.shape,
    'moments': tables.moments.shape,
}

integrated stencil 1/3: R=37.85929648
integrated stencil 2/3: R=38
integrated stencil 3/3: R=38.14070352


{'H': (3, 6, 6),
 'dH/dOmega': (3, 6, 6),
 'omega Lz': (3, 6, 6),
 'moments': (3, 8, 6, 6)}

In [31]:
# Step 5 — Add i Gamma growth widths to the diagonal entries of the six basis states
nonhermitian_hamiltonian, widths = build_nonhermitian_hamiltonians(
    state, tables
)
{
    'basis': ['BH1:200', 'BH1:21-1', 'BH1:211',
              'BH2:200', 'BH2:21-1', 'BH2:211'],
    'Gamma [Planck^-1]': widths,
    'Gamma/(mu alpha1^2)': widths / state.energy_scale,
}

{'basis': ['BH1:200', 'BH1:21-1', 'BH1:211', 'BH2:200', 'BH2:21-1', 'BH2:211'],
 'Gamma [Planck^-1]': array([-1.36581197e-46, -9.84000128e-51,  2.92011481e-51, -1.29890594e-46,
        -9.03464415e-51,  2.71469820e-51]),
 'Gamma/(mu alpha1^2)': array([-4.99375000e-04, -3.59775047e-08,  1.06766698e-08, -4.74912484e-04,
        -3.30329177e-08,  9.92561528e-09])}

In [32]:
# Step 6 — Initialize by -|E|, then track continuously using the v2 Python Hungarian-overlap method
eigensystem = diagonalize_and_track(nonhermitian_hamiltonian, widths)
{
    'E/(mu alpha1^2) at R': eigensystem.eigenvalues[1] / state.energy_scale,
    'pointwise permutations': eigensystem.permutations,
}

{'E/(mu alpha1^2) at R': array([-0.16361154+1.04552610e-08j, -0.16152054+9.00288925e-09j,
        -0.1575334 -4.99373866e-04j, -0.15544059-4.74912192e-04j,
        -0.15156285-3.62071490e-08j, -0.14947161-3.30850348e-08j]),
 'pointwise permutations': array([[0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5]])}

In [33]:
# Step 7 — Construct biorthogonal left eigenvectors and check the eigen-residual and <L|R>=I
assert eigensystem.eigen_residual_max < 1e-10
assert eigensystem.biorthogonality_max_error < 1e-10
{
    'eigen residual max': eigensystem.eigen_residual_max,
    'biorthogonality max error': eigensystem.biorthogonality_max_error,
    '<L|R> at R': eigensystem.biorthogonality[1],
}

{'eigen residual max': 1.2227804252185167e-15,
 'biorthogonality max error': 2.645586055007693e-14,
 '<L|R> at R': array([[ 1.00000000e+00+1.61881831e-24j, -1.08511290e-14-5.60533445e-16j,
         -2.01864892e-17-2.98722051e-18j, -5.21339204e-15-8.16690470e-16j,
          2.32469071e-15+2.79815648e-17j,  2.99251319e-15+4.72932696e-16j],
        [-1.08511290e-14-5.60529172e-16j,  1.00000000e+00-5.29395592e-23j,
          3.03576608e-17+1.87838026e-17j,  2.26810775e-15+4.82374437e-15j,
         -2.74622989e-15+3.30123812e-17j,  2.26728358e-15-2.37257356e-16j],
        [-2.01864892e-17-2.98727345e-18j,  3.05745013e-17+1.87295925e-17j,
          1.00000000e+00+5.29395592e-23j,  3.92686509e-16-2.06452194e-16j,
         -1.32298076e-17+9.47816633e-20j, -8.08679295e-17-1.09819092e-16j],
        [-5.21327113e-15-8.16706845e-16j,  2.26810351e-15+4.82374575e-15j,
          3.93152386e-16-2.06451726e-16j,  1.00000000e+00-1.79047444e-23j,
         -4.78655501e-15+3.12639729e-16j, -6.63281482e-15-

In [34]:
# Step 8 — Compute the biorthogonal cloud center and explicitly take its real part for conservative orbital quantities
cloud = compute_cloud_observables(state, tables, eigensystem)
{
    'Xc biorthogonal at R': cloud.center_biorthogonal[1],
    'Xc conservative at R': cloud.center[1],
    'normalization at R': cloud.normalization[1],
}

{'Xc biorthogonal at R': array([-18.90415218+2.73227863e-06j,  19.09506988-1.93202466e-05j,
        -18.90368126+1.21335167e-05j,  19.09471805+8.39271935e-06j,
        -18.90438183-5.20625365e-06j,  19.09536784+8.28235432e-07j]),
 'Xc conservative at R': array([-18.90415218,  19.09506988, -18.90368126,  19.09471805,
        -18.90438183,  19.09536784]),
 'normalization at R': array([1.00029468+2.38249539e-09j, 1.00027078+1.19074482e-08j,
        1.0044267 -1.41448664e-08j, 1.00411564-3.20778896e-09j,
        1.000327  +2.35738537e-09j, 1.00030159+7.05326407e-10j])}

In [35]:
# Step 9 — Cloud energy in the laboratory frame; use the real part for conservative energy and retain the imaginary part for termination
{
    'Ecloud/(mu alpha1^2) at R': (
        cloud.cloud_energy_lab[1] / state.energy_scale
    ),
    'dEcloud/dR at R': cloud.cloud_energy_derivative[1],
}

{'Ecloud/(mu alpha1^2) at R': array([-0.14463074+9.00112731e-09j, -0.14227671+6.87682019e-09j,
        -0.14457249-4.99369247e-04j, -0.14221655-4.74912966e-04j,
        -0.14462195-3.66898143e-08j, -0.14226741-3.28677766e-08j]),
 'dEcloud/dR at R': array([1.42190702e-46+3.51256090e-52j, 1.43173555e-46+5.13936931e-52j,
        1.40864237e-46-1.09546945e-51j, 1.41811693e-46+1.34817903e-52j,
        1.41898446e-46+1.25131152e-52j, 1.42873426e-46-2.96726462e-53j])}

In [36]:
# Step 10 — Compute the GW frequency at the three stencil points
gw_frequency_squared = compute_gw_frequency_squared(
    state, tables, eigensystem, cloud
)
assert np.all(gw_frequency_squared > 0.0)
{
    'f_GW [Planck^-1] at R': np.sqrt(gw_frequency_squared[1]),
    'dimensionless f_GW at R': (
        np.sqrt(gw_frequency_squared[1]) / state.energy_scale
    ),
}

{'f_GW [Planck^-1] at R': array([5.36859280e-46, 5.36851822e-46, 5.36685832e-46, 5.36672872e-46,
        5.36832725e-46, 5.36824481e-46]),
 'dimensionless f_GW at R': array([0.00196289, 0.00196286, 0.00196226, 0.00196221, 0.00196279,
        0.00196276])}

In [37]:
# Step 11 — Compute the real cloud quadrupole, total mass quadrupole, and GR power
mass_quadrupole = compute_mass_quadrupole(state, tables, cloud)
gr_power = compute_gr_power(gw_frequency_squared, mass_quadrupole)
assert np.all(gr_power > 0.0)
{
    'Qc biorthogonal at R, selected state': (
        cloud.quadrupole_biorthogonal[1, selected_state]
    ),
    'P_GR at R for six states': gr_power[1],
}

{'Qc biorthogonal at R, selected state': array([[ 3.69535291e+02-6.08131957e-05j, -3.29037172e-16-1.99976364e-17j,
          1.21173213e-16-5.17320582e-22j],
        [-3.29037172e-16-1.99976364e-17j,  1.18324166e+01-6.17903562e-07j,
          1.45869208e-17+8.20174145e-24j],
        [ 1.21173213e-16-5.17320582e-22j,  1.45869208e-17+8.20174145e-24j,
          5.99554588e+00+4.63346563e-07j]]),
 'P_GR at R for six states': array([2.01333845e-17, 2.01728379e-17, 2.00981241e-17, 2.01363089e-17,
        2.01248356e-17, 2.01639573e-17])}

In [38]:
# Step 12 — Compute the three-point conservative system energy, centered dE_sys/dR, and orbital step duration
system_energy = compute_system_energy(
    state, tables, cloud, gw_frequency_squared
)
system_energy_derivative, delta_time = compute_step_duration(
    state, delta_R, tables, system_energy, gr_power
)
assert delta_time > 0.0
{
    'dE_sys/dR at R': system_energy_derivative[1, selected_state],
    'delta_t [Planck time]': delta_time,
    'delta_t [year]': delta_time / PLANCK_TIMES_PER_YEAR,
}

{'dE_sys/dR at R': np.float64(1.5903463459040525e+34),
 'delta_t [Planck time]': 1.1114242846104516e+50,
 'delta_t [year]': 0.18995994717308262}

In [39]:
# Step 13 — Use eigenvector weights to split 2 Im(E) into signed BH1/BH2 channels
rates = compute_signed_rates(state, eigensystem)
assert np.all(np.isfinite(rates.channels))
assert np.all(np.isfinite([
    rates.black_hole_1, rates.black_hole_2,
    rates.total, rates.eigenvalue_total,
]))
cloud_regime = (
    'cloud grows' if rates.total > 0.0
    else 'cloud terminates' if rates.total < 0.0
    else 'cloud mass is unchanged'
)
bh1_regime = (
    'BH1 emits to cloud' if rates.black_hole_1 > 0.0
    else 'BH1 absorbs from cloud' if rates.black_hole_1 < 0.0
    else 'BH1 has no cloud exchange'
)
bh2_regime = (
    'BH2 emits to cloud' if rates.black_hole_2 > 0.0
    else 'BH2 absorbs from cloud' if rates.black_hole_2 < 0.0
    else 'BH2 has no cloud exchange'
)
{
    'six channel rates': rates.channels,
    'BH1 rate': rates.black_hole_1,
    'BH2 rate': rates.black_hole_2,
    'total from channels': rates.total,
    '2 Im(E)': rates.eigenvalue_total,
    'consistency error': rates.consistency_error,
}

{'six channel rates': array([-2.28186960e-57, -3.57543807e-54,  5.83913007e-51, -1.16467050e-52,
        -1.20651603e-57,  3.30084114e-56]),
 'BH1 rate': 5.835552349865523e-51,
 'BH2 rate': -1.164352478521801e-52,
 'total from channels': 5.719117102013343e-51,
 '2 Im(E)': 5.719117103765809e-51,
 'consistency error': 1.7524651878324136e-60}

In [40]:
# Step 14 — Evolve the cloud exactly and exponentially, transferring the signed mass exchange back to both black holes
exchange = apply_mass_exchange(state, delta_R, delta_time, rates)
next_state = exchange.next_state
assert rates.total * exchange.cloud_delta_solar >= 0.0
assert next_state.spin_primary == state.spin_primary
assert next_state.spin_secondary == state.spin_secondary
assert exchange.conservation_error_solar < 1e-12
{
    'rate * delta_t': exchange.exponent,
    'delta M1 [solar]': exchange.primary_delta_solar,
    'delta M2 [solar]': exchange.secondary_delta_solar,
    'delta Mc [solar]': exchange.cloud_delta_solar,
    'mass conservation error [solar]': exchange.conservation_error_solar,
}

{'rate * delta_t': 0.6356365633708579,
 'delta M1 [solar]': -3.6252281602777865,
 'delta M2 [solar]': 0.07233322812576158,
 'delta Mc [solar]': 3.552894932152025,
 'mass conservation error [solar]': 2.0816681711721685e-16}

In [41]:
# Step 15 — Results of one complete iteration; do not continue to another step
total_before = (
    state.primary_mass_solar + state.secondary_mass_solar
    + state.cloud_mass_solar
)
total_after = (
    next_state.primary_mass_solar + next_state.secondary_mass_solar
    + next_state.cloud_mass_solar
)
assert np.isclose(total_before, total_after, rtol=0.0, atol=1e-12)
{
    'current state': state.summary(),
    'next state': next_state.summary(),
    'cloud mass factor': (
        next_state.cloud_mass_solar / state.cloud_mass_solar
    ),
    'total mass before/after [solar]': (total_before, total_after),
    'interpretation': f'{bh1_regime}; {bh2_regime}; {cloud_regime}',
}

{'current state': {'M1_solar': 40.0,
  'M2_solar': 39.6,
  'Mc_solar': 4.0,
  'q': 0.99,
  'qc': 0.1,
  'alpha1': 0.09999999999999999,
  'alpha2': 0.099,
  'a1': 1.0,
  'a2': 1.0,
  'R': 38.0,
  'state': 1},
 'next state': {'M1_solar': 36.374771839722214,
  'M2_solar': 39.672333228125765,
  'Mc_solar': 7.552894932152025,
  'q': 1.090655177245745,
  'qc': 0.20764102563810624,
  'alpha1': 0.09093692959930552,
  'alpha2': 0.09918083307031442,
  'a1': 1.0,
  'a2': 1.0,
  'R': 37.85929648241206,
  'state': 1},
 'cloud mass factor': 1.8882237330380063,
 'total mass before/after [solar]': (83.6, 83.60000000000001),
 'interpretation': 'BH1 emits to cloud; BH2 absorbs from cloud; cloud grows'}